# 03 · Phenology metrics per year and topographic zone

Loads the cached Sentinel-2 statistics, filters dates by quality and derives the autumn phenology of every year and zone.

- **Settings:** `config.py` (years, date filters, smoothing and fit settings)
- **Reads:** `cache/<AOI>_zones/` (from `01_download_sentinel2`)
- **Writes** to `results/<AOI>/`:
  - `eos_by_zone.csv` and `logistic_by_zone.csv`: inputs of `04_climate_analysis`
  - `zones.csv`, `topography_effects.csv`, `eos_projection.csv`
  - `ndvi_timeseries_all_dates.csv`, `ndvi_timeseries_clean.csv` and `ndvi_monthly.csv`

Runs offline in seconds.

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.optimize import curve_fit
from scipy.stats import kendalltau, linregress, theilslopes
from statsmodels.tsa.seasonal import STL

from config import *
from phenology_utils import clean_frame, index_frame, load_stats, load_zone_table

INDEX_COLORS = {"NDVI": "darkgreen", "GRVI": "darkorange", "PSRI": "purple"}
ASPECT_COLORS = {"N": "tab:blue", "S": "tab:orange", "flat": "grey"}
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | results: {OUT_DIR}")

## 1. Load and filter the cached statistics

A date is kept for a zone if at least `MIN_VALID_FRACTION` of the zone's pixels are clear (and it passes `MAX_SCENE_CLOUD` / `MAX_SCL_FRACTION`). `df`, `df_clean` and so on refer to the whole AOI (zone `"all"`). `clean_by_zone[index][zone]` holds the cleaned median series of every zone.

In [ ]:
stats_all = load_stats()
zone_table = load_zone_table()
ZONES = list(zone_table.index)   # "all" first, then elevation band x aspect

frames = {name: index_frame(stats_all, name.lower()) for name in INDICES}
clean = {name: clean_frame(stats_all, name.lower()) for name in INDICES}
df, df_clean = frames["NDVI"], clean["NDVI"]
for name in INDICES:
    print(f"{name}: {len(clean[name])} of {len(frames[name])} dates kept")

clean_by_zone = {name: {zone: clean_frame(stats_all, name.lower(), zone)[f"{name.lower()}_median"] for zone in ZONES}
                 for name in INDICES}
display(zone_table[["n_pixels", "elev_mean", "slope_mean", "northness_mean"]].round(2).join(
    pd.DataFrame({name: {z: len(s) for z, s in by_zone.items()} for name, by_zone in clean_by_zone.items()})
      .add_prefix("dates_")))

## 2. Composite and smooth (whole AOI)

Shows monthly median composites and a centred moving average (`SMOOTH_WINDOW` × 5 days) on a regular 5-day grid, with gaps of up to 30 days interpolated.

In [ ]:
def smooth(series):
    regular = series.resample("5D").mean().interpolate(method="time", limit=6)
    rolled = regular.rolling(SMOOTH_WINDOW, center=True, min_periods=3).mean()
    return rolled.where(regular.notna())


medians = {name: clean[name][f"{name.lower()}_median"] for name in INDICES}
monthly = {name: s.resample("MS").median() for name, s in medians.items()}
smoothed = {name: smooth(s) for name, s in medians.items()}

fig, ax = plt.subplots(figsize=(12, 4))
for name, s in smoothed.items():
    ax.plot(s.index, s, lw=2, color=INDEX_COLORS[name], label=name)
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("index value")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
ax.set_title(f"NDVI, GRVI and PSRI ({SMOOTH_WINDOW * 5}-day moving average)")
plt.show()

ylims = {"NDVI": (-0.1, 1), "GRVI": (-1, 1), "PSRI": None}
for name in INDICES:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(frames[name].index, frames[name][f"{name.lower()}_median"], s=10, c="lightgrey", label="all dates")
    ax.scatter(medians[name].index, medians[name], s=14, label="clear dates")
    ax.plot(monthly[name].index, monthly[name], "o--", label="monthly median")
    ax.plot(smoothed[name].index, smoothed[name], lw=2, label=f"{SMOOTH_WINDOW * 5}-day moving average")
    ax.set_ylabel(name)
    if ylims[name]:
        ax.set_ylim(*ylims[name])
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_title(name)
    plt.show()

## 3. End of season (EOS) from smoothed NDVI

For every year, EOS is the first date after the year's peak at which the smoothed NDVI drops to
- **fixed:** `EOS_FIXED_THRESHOLD`, or
- **midpoint:** halfway between the peak and the minimum after the peak.

Years with fewer than `MIN_POINTS_PER_YEAR` values, or where no crossing occurs (e.g. snow hides the decline), get no EOS. `eos_table` covers all zones. The plot shows the whole AOI, with a linear trend in days per year.

In [ ]:
def first_crossing(post_peak, threshold):
    below = post_peak[post_peak <= threshold]
    return below.index.min() if not below.empty else pd.NaT


def eos_rows(series):
    # One EOS row per year of a smoothed NDVI series
    rows = []
    s_all = series.dropna().sort_index()
    for year, s in s_all.groupby(s_all.index.year):
        if len(s) < MIN_POINTS_PER_YEAR:
            continue
        peak = s.idxmax()
        post_peak = s[s.index >= peak]
        peak_value = s.loc[peak]
        midpoint = post_peak.min() + 0.5 * (peak_value - post_peak.min())
        eos_fixed = first_crossing(post_peak, EOS_FIXED_THRESHOLD)
        eos_mid = first_crossing(post_peak, midpoint)
        rows.append({
            "year": year,
            "n_points": len(s),
            "peak_date": peak.date(),
            "peak_ndvi": round(peak_value, 3),
            "midpoint_threshold": round(midpoint, 3),
            "eos_fixed_date": eos_fixed.date() if pd.notna(eos_fixed) else None,
            "eos_fixed_doy": eos_fixed.dayofyear if pd.notna(eos_fixed) else np.nan,
            "eos_mid_date": eos_mid.date() if pd.notna(eos_mid) else None,
            "eos_mid_doy": eos_mid.dayofyear if pd.notna(eos_mid) else np.nan,
        })
    return rows


rows = [{"zone": zone, **row} for zone in ZONES for row in eos_rows(smooth(clean_by_zone["NDVI"][zone]))]
eos_table = pd.DataFrame(rows).set_index(["zone", "year"])
eos_all = eos_table.xs("all", level="zone")
display(eos_all)

EOS_METHODS = [("eos_fixed_doy", f"EOS fixed ({EOS_FIXED_THRESHOLD})", "tab:blue"),
               ("eos_mid_doy", "EOS midpoint", "tab:red")]
fig, ax = plt.subplots(figsize=(10, 4))
for col, label, color in EOS_METHODS:
    d = eos_all[col].dropna()
    ax.plot(d.index, d.values, "o", color=color, label=label)
    if len(d) >= 3:
        fit = linregress(d.index.values, d.values)
        x = np.array([d.index.min(), d.index.max()])
        ax.plot(x, fit.intercept + fit.slope * x, "--", color=color,
                label=f"trend {fit.slope:+.2f} d/yr (p={fit.pvalue:.2f})")
ax.set_xlabel("year")
ax.set_ylabel("EOS day of year")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("End of season per year (smoothed NDVI, whole AOI)")
plt.show()

### EOS trend test and short-term projection (no climate drivers)

- **Mann-Kendall / Theil-Sen:** a robust check for a monotonic trend. Theil-Sen gives the slope with a 95 % CI; Mann-Kendall gives the p-value.
- **Projection:** an OLS trend extrapolated `FORECAST_YEARS` ahead, with a 95 % prediction interval. A mean-only model is shown alongside. If the trend is not significant, the mean ± interval is the more honest expectation.

This is the statistical baseline that the climate models in `04` are compared against.

In [ ]:
proj_rows = []
fig, axes = plt.subplots(1, len(EOS_METHODS), figsize=(14, 4.5), sharey=True)
for ax, (col, label, color) in zip(axes, EOS_METHODS):
    d = eos_all[col].dropna()
    n = len(d)
    if n < 4:
        ax.set_title(f"{label}: only {n} years, skipped")
        continue
    years = d.index.values.astype(float)
    y = d.values.astype(float)
    ts_slope, ts_int, ts_lo, ts_hi = theilslopes(y, years, alpha=1 - ALPHA)
    tau, mk_p = kendalltau(years, y)

    x0 = years.mean()   # centred years for numerical stability
    ols = sm.OLS(y, sm.add_constant(years - x0)).fit()
    mean_model = sm.OLS(y, np.ones((n, 1))).fit()
    future = np.arange(years.max() + 1, years.max() + 1 + FORECAST_YEARS)
    grid = np.arange(years.min(), future.max() + 1)
    pred = ols.get_prediction(sm.add_constant(grid - x0, has_constant="add")).summary_frame(alpha=ALPHA)
    pred_mean = mean_model.get_prediction(np.ones((len(grid), 1))).summary_frame(alpha=ALPHA)

    print(f"{label} (n={n} years)")
    print(f"  Theil-Sen slope: {ts_slope:+.2f} d/yr  (95% CI {ts_lo:+.2f} to {ts_hi:+.2f})")
    print(f"  Mann-Kendall: tau={tau:+.2f}, p={mk_p:.3f}")
    print(f"  OLS slope: {ols.params[1]:+.2f} d/yr, p={ols.pvalues[1]:.3f}, R2={ols.rsquared:.2f}")
    for i, yr in enumerate(grid):
        if yr in future:
            proj_rows.append({
                "method": label, "year": int(yr),
                "trend_doy": pred["mean"].iloc[i],
                "trend_pi_low": pred["obs_ci_lower"].iloc[i], "trend_pi_high": pred["obs_ci_upper"].iloc[i],
                "mean_doy": pred_mean["mean"].iloc[i],
                "mean_pi_low": pred_mean["obs_ci_lower"].iloc[i], "mean_pi_high": pred_mean["obs_ci_upper"].iloc[i],
                "trend_significant": bool(mk_p < ALPHA),
            })

    ax.fill_between(grid, pred["obs_ci_lower"], pred["obs_ci_upper"], color=color, alpha=0.15,
                    label="OLS 95% prediction interval")
    ax.fill_between(grid, pred["mean_ci_lower"], pred["mean_ci_upper"], color=color, alpha=0.3,
                    label="OLS 95% CI of the trend")
    ax.plot(grid, pred["mean"], color=color, lw=2, label=f"OLS trend {ols.params[1]:+.2f} d/yr")
    ax.plot(grid, ts_int + ts_slope * grid, ":", color="black", lw=1.5,
            label=f"Theil-Sen {ts_slope:+.2f} d/yr (MK p={mk_p:.2f})")
    ax.plot(grid, pred_mean["mean"], "--", color="grey", lw=1, label="mean (no trend)")
    ax.fill_between(grid, pred_mean["obs_ci_lower"], pred_mean["obs_ci_upper"], facecolor="none",
                    edgecolor="grey", hatch="//", lw=0, alpha=0.5, label="mean 95% prediction interval")
    ax.plot(years, y, "o", color=color, label="observed EOS")
    ax.axvspan(future.min() - 0.5, future.max() + 0.5, color="grey", alpha=0.08)
    ax.set_title(label)
    ax.set_xlabel("year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("EOS day of year")
axes[0].legend(fontsize=8, loc="best")
plt.suptitle("EOS trend and projection (shaded columns = projected years)")
plt.tight_layout()
plt.show()

eos_projection = pd.DataFrame(proj_rows)
if not eos_projection.empty:
    eos_projection = eos_projection.set_index(["method", "year"]).round(1)
    for c in ["trend_doy", "mean_doy"]:
        eos_projection[c.replace("_doy", "_date")] = [date(int(yr), 1, 1) + timedelta(days=int(round(v)) - 1)
                                                      for (_, yr), v in eos_projection[c].items()]
    display(eos_projection)

## 4. Per-year logistic fit of the autumn transition (raw clear observations)

For each year, index and zone, a logistic curve is fitted to the raw clear observations within `FIT_WINDOW` (day of year):

`y(t) = lo + (hi - lo) / (1 + exp((t - t0) / s))`

- `hi` is the summer level, `lo` the post-transition level, `t0` the 50 % point and `s` the time scale in days.
- NDVI and GRVI drop (`hi > lo`); PSRI rises (`hi < lo`).
- **onset / mid / end** = 20 / 50 / 80 % of the transition: `t = t0 + s · ln(f / (1 − f))`.

Quality columns:
- `n_obs`: clear observations used in the fit
- `n_before_mid` / `n_after_mid`: observations on each side of `t0`
- `rmse`: fit error
- `direction_ok`: the transition goes the expected way for that index

The plots show the whole AOI.

In [ ]:
def logistic(t, hi, lo, t0, s):
    return lo + (hi - lo) / (1 + np.exp((t - t0) / s))


def fit_logistic(doy, y):
    split = np.median(doy)
    p0 = [np.median(y[doy <= split]), np.median(y[doy > split]), 280, 7]
    bounds = ([-2, -2, FIT_WINDOW[0], 1], [2, 2, FIT_WINDOW[1], 40])
    popt, _ = curve_fit(logistic, doy, y, p0=p0, bounds=bounds, maxfev=20000)
    return popt


def logistic_rows(name, series, zone="all"):
    # Per-year fit of one index's clear observations; returns the table rows and {year: (doy, y, popt)}
    rows, fits = [], {}
    ser = series.dropna()
    for year, s in ser.groupby(ser.index.year):
        doy = s.index.dayofyear.values.astype(float)
        sel = (doy >= FIT_WINDOW[0]) & (doy <= FIT_WINDOW[1])
        doy, y = doy[sel], s.values[sel]
        if len(y) == 0:
            continue
        row = {"index": name, "zone": zone, "year": year, "n_obs": len(y)}
        popt = None
        if len(y) >= MIN_OBS_PER_YEAR:
            try:
                popt = fit_logistic(doy, y)
            except RuntimeError:
                print(f"{name} {zone} {year}: fit did not converge")
        if popt is not None:
            hi, lo, t0, sc = popt
            row.update({
                "hi": hi, "lo": lo, "scale_days": sc,
                "rmse": np.sqrt(np.mean((logistic(doy, *popt) - y) ** 2)),
                "n_before_mid": int((doy < t0).sum()),
                "n_after_mid": int((doy >= t0).sum()),
                "direction_ok": bool((hi > lo) == EXPECT_DECREASE[name]),
            })
            for label, f in TRANSITION_FRACTIONS.items():
                d = t0 + sc * np.log(f / (1 - f))
                row[f"{label}_doy"] = d
                row[f"{label}_date"] = date(int(year), 1, 1) + timedelta(days=int(round(d)) - 1)
        fits[year] = (doy, y, popt)
        rows.append(row)
    return rows, fits


rows, fits = [], {}
for name in INDICES:
    for zone in ZONES:
        zone_rows, zone_fits = logistic_rows(name, clean_by_zone[name][zone], zone)
        rows += zone_rows
        if zone == "all":
            fits.update({(name, year): fit for year, fit in zone_fits.items()})
logistic_table = pd.DataFrame(rows).set_index(["index", "zone", "year"]).round(3)
display(logistic_table.xs("all", level="zone"))

fit_years = sorted({yr for _, yr in fits})
tt = np.linspace(*FIT_WINDOW, 300)
fig, axes = plt.subplots(len(INDICES), len(fit_years), squeeze=False, sharex=True,
                         figsize=(2.4 * len(fit_years), 2.4 * len(INDICES)))
for i, name in enumerate(INDICES):
    for j, year in enumerate(fit_years):
        ax = axes[i, j]
        if (name, year) in fits:
            doy, y, popt = fits[(name, year)]
            ax.plot(doy, y, "o", ms=3, color=INDEX_COLORS[name])
            if popt is not None:
                ax.plot(tt, logistic(tt, *popt), color="black", lw=1)
                for label, f in TRANSITION_FRACTIONS.items():
                    ax.axvline(popt[2] + popt[3] * np.log(f / (1 - f)), color="red",
                               lw=1.2 if label == "mid" else 0.6, ls="--")
        if i == 0:
            ax.set_title(str(year))
        if j == 0:
            ax.set_ylabel(name)
        if i == len(INDICES) - 1:
            ax.set_xlabel("DOY")
        ax.grid(alpha=0.3)
plt.suptitle("Per-year logistic fit of the autumn transition (red dashed: onset / mid / end)")
plt.tight_layout()
plt.show()

## 5. Phenology vs topography (elevation band × aspect)

Only logistic fits with `direction_ok` and at least `MIN_OBS_SIDE` observations on each side of `t0` are used.

- **Plot:** transition DOY against elevation band, one line per aspect (mean ± sd over years).
- **Model:** OLS `doy ~ elev_100m + C(aspect) + C(year)`.
  - The elevation coefficient is in days per 100 m.
  - The aspect coefficient is the north − south difference in days.
  - `C(year)` absorbs year-to-year weather.

**Caveats:**
- Zones share weather and clouds within a year, so the CIs are somewhat optimistic.
- North slopes lose more autumn dates to SCL class 2 (dark / topographic shadow); see `mean_n_obs_*` and `autumn_dark_fraction`.
- PSRI on north slopes in late autumn (low sun) is the most affected.

In [ ]:
autumn = stats_all[stats_all.index.dayofyear >= FIT_WINDOW[0]]
coverage = (zone_table[["elev_band", "aspect", "n_pixels", "elev_mean", "slope_mean", "northness_mean"]]
            .join(logistic_table.groupby(level=["zone", "index"]).n_obs.mean().unstack().add_prefix("mean_n_obs_"))
            .join(autumn.groupby("zone").scl_dark_fraction.mean().rename("autumn_dark_fraction")))
display(coverage.round(2))

zone_attrs = zone_table.drop(index="all")[["elev_band", "aspect", "elev_mean"]]
fits_ok = logistic_table.reset_index().join(zone_attrs, on="zone", how="inner")
fits_ok = fits_ok[fits_ok.direction_ok.eq(True) & (fits_ok.n_before_mid >= MIN_OBS_SIDE)
                  & (fits_ok.n_after_mid >= MIN_OBS_SIDE)]
eos_zones = eos_table.reset_index().join(zone_attrs, on="zone", how="inner")

responses = [(f"{name} logistic mid", fits_ok[fits_ok["index"] == name], "mid_doy") for name in INDICES]
responses.append(("NDVI EOS midpoint", eos_zones, "eos_mid_doy"))

fig, axes = plt.subplots(1, len(responses), figsize=(4 * len(responses), 4), sharex=True)
for ax, (label, data, col) in zip(axes, responses):
    for asp, color in ASPECT_COLORS.items():
        d = data[data.aspect == asp].dropna(subset=[col])
        if d.empty:
            continue
        shift = ELEV_STEP / 2 + {"N": -0.08, "S": 0.08, "flat": 0}[asp] * ELEV_STEP   # band centre, dodged
        ax.scatter(d.elev_band + shift, d[col], s=8, color=color, alpha=0.3)
        g = d.groupby("elev_band")[col].agg(["mean", "std"])
        ax.errorbar(g.index + shift, g["mean"], yerr=g["std"].fillna(0), fmt="o-", color=color, capsize=3,
                    label=asp)
    ax.set_title(label)
    ax.set_xlabel("elevation band centre (m)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("day of year")
axes[0].legend(title="aspect")
plt.suptitle("Autumn transition vs elevation and aspect (points: zone-years, lines: mean ± sd over years)")
plt.tight_layout()
plt.show()

NS_TERM = "C(aspect, Treatment('S'))[T.N]"


def topo_model(data, col):
    d = data.dropna(subset=[col]).assign(elev_100m=lambda x: x.elev_mean / 100)
    if (not {"N", "S"} <= set(d.aspect) or d.elev_band.nunique() < 2
            or len(d) < d.year.nunique() + d.aspect.nunique() + 3):
        return None
    return smf.ols(f"{col} ~ elev_100m + C(aspect, Treatment('S')) + C(year)", data=d).fit()


rows = []
for label, data, col in responses:
    fit = topo_model(data, col)
    if fit is None:
        print(f"{label}: too few zone-years (needs N and S zones and >= 2 elevation bands), no model")
        continue
    ci = fit.conf_int(ALPHA)
    rows.append({
        "response": label, "n_zone_years": int(fit.nobs), "n_zones": data.zone.nunique(),
        "days_per_100m": fit.params["elev_100m"],
        "elev_ci_low": ci.loc["elev_100m", 0], "elev_ci_high": ci.loc["elev_100m", 1],
        "elev_p": fit.pvalues["elev_100m"],
        "north_minus_south_days": fit.params[NS_TERM],
        "ns_ci_low": ci.loc[NS_TERM, 0], "ns_ci_high": ci.loc[NS_TERM, 1], "ns_p": fit.pvalues[NS_TERM],
        "r2": fit.rsquared,
    })
topo_effects = pd.DataFrame(rows).set_index("response").round(3) if rows else pd.DataFrame()
display(topo_effects)

## 6. Trend / seasonal / remainder decomposition (STL, whole AOI)

Robust STL on a regular 5-day grid splits each index into a long-term trend, a repeating yearly cycle and the remainder. It needs at least two full years.

In [ ]:
STEP_DAYS = 5
PERIOD = int(round(365.25 / STEP_DAYS))   # observations per year on the regular grid


def stl_decompose(series, name):
    s = series.resample(f"{STEP_DAYS}D").mean()
    s = s.loc[s.first_valid_index():s.last_valid_index()].interpolate(method="time")
    if len(s) < 2 * PERIOD:
        print(f"{name}: {len(s) * STEP_DAYS / 365.25:.1f} years of data; STL needs at least 2 full years")
        return None
    res = STL(s, period=PERIOD, robust=True).fit()
    return pd.DataFrame({"observed": s, "trend": res.trend, "seasonal": res.seasonal, "remainder": res.resid})


for name in INDICES:
    dec = stl_decompose(medians[name], name)
    if dec is None:
        continue
    color = INDEX_COLORS[name]
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(dec.index, dec.observed, color="grey", lw=1)
    axes[0].set_ylabel(f"{name} observed")
    axes[1].plot(dec.index, dec.trend, color=color, lw=2)
    axes[1].set_ylabel("trend")
    axes[2].plot(dec.index, dec.seasonal, color=color, lw=1.5)
    axes[2].set_ylabel("seasonal")
    axes[3].scatter(dec.index, dec.remainder, s=6, color=color)
    axes[3].set_ylabel("remainder")
    for a in axes[2:]:
        a.axhline(0, color="black", lw=0.5)
    for a in axes:
        a.grid(alpha=0.3)
    axes[0].set_title(f"{name} STL decomposition (period = {PERIOD} x {STEP_DAYS} days)")
    plt.tight_layout()
    plt.show()
    slope = np.polyfit((dec.index - dec.index[0]).days / 365.25, dec.trend.values, 1)[0]
    print(f"{name} trend slope: {slope:+.4f} per year | seasonal amplitude: "
          f"{dec.seasonal.max() - dec.seasonal.min():.3f} | remainder std: {dec.remainder.std():.3f}")

## 7. Export

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / "ndvi_timeseries_all_dates.csv")
df_clean.to_csv(OUT_DIR / "ndvi_timeseries_clean.csv")
monthly["NDVI"].to_frame("ndvi_monthly_median").to_csv(OUT_DIR / "ndvi_monthly.csv")
eos_table.to_csv(OUT_DIR / "eos_by_zone.csv")
logistic_table.to_csv(OUT_DIR / "logistic_by_zone.csv")
zone_table.to_csv(OUT_DIR / "zones.csv")
topo_effects.to_csv(OUT_DIR / "topography_effects.csv")
eos_projection.to_csv(OUT_DIR / "eos_projection.csv")
print(f"Results written to {OUT_DIR}")